# Analisis Performa e2ee_sim
Notebook ini digunakan untuk menganalisis dan memvisualisasikan hasil pengujian performa program simulasi e2ee_sim, mencakup perbandingan AES-GCM-SIV vs AES-GCM pada seluruh kategori data uji.

## 1. Import dan Konfigurasi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# Konfigurasi tampilan grafik
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Warna konsisten untuk kedua algoritma
COLORS = {
    'aes-gcm-siv': '#2563EB',  # biru
    'aes-gcm':     '#DC2626',  # merah
}
LABELS = {
    'aes-gcm-siv': 'AES-GCM-SIV (usulan)',
    'aes-gcm':     'AES-GCM (baseline)',
}

# Folder output grafik
os.makedirs('grafik', exist_ok=True)

print('Konfigurasi selesai.')

## 2. Load Data CSV
Ganti path file CSV sesuai lokasi file hasil e2ee_sim.

In [ ]:
# ============================================================
# GANTI PATH INI SESUAI LOKASI FILE CSV KAMU
CSV_PATH = 'performance.csv'
# ============================================================

df = pd.read_csv(CSV_PATH)

# Normalisasi nama algoritma ke lowercase agar cocok dengan COLORS/LABELS
df['algorithm'] = df['algorithm'].str.lower()

print(f'Total baris: {len(df)}')
print(f'Kolom: {df.columns.tolist()}')
print(f'\nAlgoritma: {df["algorithm"].unique()}')
print(f'Tipe data: {df["data_type"].unique()}')
print(f'\nPreview:')
df.head()

## 3. Fungsi Bantu

In [ ]:
def hitung_statistik(df, group_cols=['algorithm', 'data_type']):
    """Hitung rata-rata dan standar deviasi per grup."""
    return df.groupby(group_cols).agg(
        n=('iteration', 'count'),
        data_size_bytes=('data_size_bytes', 'first'),
        enc_mean=('enc_time_us', 'mean'),
        enc_std=('enc_time_us', 'std'),
        dec_mean=('dec_time_us', 'mean'),
        dec_std=('dec_time_us', 'std'),
        enc_tp_mean=('enc_throughput_mbs', 'mean'),
        enc_tp_std=('enc_throughput_mbs', 'std'),
        dec_tp_mean=('dec_throughput_mbs', 'mean'),
        dec_tp_std=('dec_throughput_mbs', 'std'),
        overhead_bytes=('overhead_bytes', 'first'),
        overhead_pct=('overhead_pct', 'first'),
    ).round(4).reset_index()


def ttest_per_datatype(df, metric='enc_time_us'):
    """Jalankan t-test independen per data_type untuk metrik tertentu."""
    hasil = []
    for dt in df['data_type'].unique():
        gcm   = df[(df['algorithm'] == 'aes-gcm')     & (df['data_type'] == dt)][metric]
        siv   = df[(df['algorithm'] == 'aes-gcm-siv') & (df['data_type'] == dt)][metric]
        if len(gcm) == 0 or len(siv) == 0:
            continue
        t, p  = stats.ttest_ind(gcm, siv)
        hasil.append({
            'data_type': dt,
            'metric': metric,
            'mean_gcm': gcm.mean(),
            'mean_siv': siv.mean(),
            'selisih': siv.mean() - gcm.mean(),
            'selisih_pct': ((siv.mean() - gcm.mean()) / gcm.mean()) * 100,
            't_stat': round(t, 4),
            'p_value': round(p, 6),
            'signifikan': 'Ya' if p < 0.05 else 'Tidak',
        })
    return pd.DataFrame(hasil)


def simpan_grafik(nama_file):
    """Simpan grafik ke folder grafik/."""
    path = f'grafik/{nama_file}'
    plt.savefig(path, bbox_inches='tight', facecolor='white')
    print(f'Grafik disimpan: {path}')


print('Fungsi bantu siap.')

## 4. Statistik Deskriptif

In [ ]:
stats_df = hitung_statistik(df)

print('=== RINGKASAN STATISTIK ===')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
stats_df

In [ ]:
ringkasan_metrik = (
    df.groupby(['algorithm', 'data_type'])
    .agg(
        enc_mean=('enc_time_us', 'mean'),
        enc_std=('enc_time_us', 'std'),
        dec_mean=('dec_time_us', 'mean'),
        dec_std=('dec_time_us', 'std'),
        tp_enc_mean=('enc_throughput_mbs', 'mean'),
        tp_enc_std=('enc_throughput_mbs', 'std'),
        tp_dec_mean=('dec_throughput_mbs', 'mean'),
        tp_dec_std=('dec_throughput_mbs', 'std'),
        data_size_bytes=('data_size_bytes', 'first'),
        memory_mb=('memory_mb', 'mean'),
        overhead_bytes=('overhead_bytes', 'first'),
    )
    .round(4)
    .reset_index()
)

ringkasan_metrik = ringkasan_metrik.rename(columns={
    'data_type': 'Kategori',
    'algorithm': 'Algoritma',
    'enc_mean': 'Enc (µs)',
    'enc_std': 'Std Dev Enc',
    'dec_mean': 'Dec (µs)',
    'dec_std': 'Std Dev Dec',
    'tp_enc_mean': 'TP Enc (MB/s)',
    'tp_enc_std': 'Std Dev TP Enc',
    'tp_dec_mean': 'TP Dec (MB/s)',
    'tp_dec_std': 'Std Dev TP Dec',
    'data_size_bytes': 'Ukuran Data (bytes)',
    'memory_mb': 'Memori (MB)',
    'overhead_bytes': 'Overhead (byte)',
})

ringkasan_metrik = ringkasan_metrik[[
    'Kategori',
    'Algoritma',
    'Enc (µs)',
    'Std Dev Enc',
    'Dec (µs)',
    'Std Dev Dec',
    'TP Enc (MB/s)',
    'Std Dev TP Enc',
    'TP Dec (MB/s)',
    'Std Dev TP Dec',
    'Ukuran Data (bytes)',
    'Memori (MB)',
    'Overhead (byte)',
]]

print('=== RINGKASAN METRIK ===')
ringkasan_metrik

## 5. Uji T-Test Independen

In [ ]:
print('=== T-TEST: WAKTU ENKRIPSI ===')
tt_enc = ttest_per_datatype(df, 'enc_time_us')
print(tt_enc.to_string(index=False))

print('\n=== T-TEST: WAKTU DEKRIPSI ===')
tt_dec = ttest_per_datatype(df, 'dec_time_us')
print(tt_dec.to_string(index=False))

print('\n=== T-TEST: THROUGHPUT ENKRIPSI ===')
tt_tp = ttest_per_datatype(df, 'enc_throughput_mbs')
print(tt_tp.to_string(index=False))

## 6. Grafik per Kategori Data
### 6.1 Filter Data per Kategori

In [ ]:
# ============================================================
# Sesuaikan nilai data_type dengan yang ada di CSV kamu
# Contoh untuk teks:
KATEGORI_TEKS   = ['text_short', 'text_medium', 'text_long']
KATEGORI_GAMBAR = ['image_small', 'image_medium', 'image_large']
KATEGORI_AUDIO  = ['audio_small', 'audio_medium', 'audio_large']
KATEGORI_VIDEO  = ['video_small', 'video_medium', 'video_large']
# ============================================================

def filter_kategori(df, kategori):
    tersedia = [k for k in kategori if k in df['data_type'].unique()]
    if not tersedia:
        return None
    return df[df['data_type'].isin(tersedia)]

df_teks   = filter_kategori(df, KATEGORI_TEKS)
df_gambar = filter_kategori(df, KATEGORI_GAMBAR)
df_audio  = filter_kategori(df, KATEGORI_AUDIO)
df_video  = filter_kategori(df, KATEGORI_VIDEO)

for nama, data in [('Teks', df_teks), ('Gambar', df_gambar), 
                   ('Audio', df_audio), ('Video', df_video)]:
    if data is not None:
        print(f'{nama}: {len(data)} baris, tipe: {data["data_type"].unique()}')
    else:
        print(f'{nama}: belum ada data')

### 6.2 Fungsi Grafik Utama

In [ ]:
def grafik_waktu_enkripsi_dekripsi(data, label_x, judul_kategori, nama_file):
    """Grafik batang waktu enkripsi dan dekripsi per ukuran data."""
    if data is None:
        print(f'Data {judul_kategori} belum tersedia, grafik dilewati.')
        return

    stat = hitung_statistik(data)
    data_types = data['data_type'].unique()
    x = np.arange(len(data_types))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Perbandingan Waktu Enkripsi dan Dekripsi\n{judul_kategori}', 
                 fontsize=13, fontweight='bold', y=1.02)

    for ax, metrik, metrik_std, judul_ax, satuan in [
        (axes[0], 'enc_mean', 'enc_std', 'Waktu Enkripsi', 'µs'),
        (axes[1], 'dec_mean', 'dec_std', 'Waktu Dekripsi', 'µs'),
    ]:
        for i, algo in enumerate(['aes-gcm-siv', 'aes-gcm']):
            sub = stat[stat['algorithm'] == algo].set_index('data_type')
            vals = [sub.loc[dt, metrik] if dt in sub.index else 0 for dt in data_types]
            errs = [sub.loc[dt, metrik_std] if dt in sub.index else 0 for dt in data_types]
            offset = (i - 0.5) * width
            bars = ax.bar(x + offset, vals, width, 
                          label=LABELS[algo], color=COLORS[algo],
                          alpha=0.85, yerr=errs, capsize=4,
                          error_kw={'linewidth': 1.2})

        ax.set_xlabel('Kategori Ukuran Data')
        ax.set_ylabel(f'Waktu Rata-rata ({satuan})')
        ax.set_title(judul_ax)
        ax.set_xticks(x)
        ax.set_xticklabels(label_x)
        ax.legend()

    plt.tight_layout()
    simpan_grafik(nama_file)
    plt.show()


def grafik_throughput(data, label_x, judul_kategori, nama_file):
    """Grafik batang throughput enkripsi dan dekripsi per ukuran data."""
    if data is None:
        print(f'Data {judul_kategori} belum tersedia, grafik dilewati.')
        return

    stat = hitung_statistik(data)
    data_types = data['data_type'].unique()
    x = np.arange(len(data_types))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Perbandingan Throughput Enkripsi dan Dekripsi\n{judul_kategori}',
                 fontsize=13, fontweight='bold', y=1.02)

    for ax, metrik, metrik_std, judul_ax in [
        (axes[0], 'enc_tp_mean', 'enc_tp_std', 'Throughput Enkripsi'),
        (axes[1], 'dec_tp_mean', 'dec_tp_std', 'Throughput Dekripsi'),
    ]:
        for i, algo in enumerate(['aes-gcm-siv', 'aes-gcm']):
            sub = stat[stat['algorithm'] == algo].set_index('data_type')
            vals = [sub.loc[dt, metrik] if dt in sub.index else 0 for dt in data_types]
            errs = [sub.loc[dt, metrik_std] if dt in sub.index else 0 for dt in data_types]
            offset = (i - 0.5) * width
            ax.bar(x + offset, vals, width,
                   label=LABELS[algo], color=COLORS[algo],
                   alpha=0.85, yerr=errs, capsize=4,
                   error_kw={'linewidth': 1.2})

        ax.set_xlabel('Kategori Ukuran Data')
        ax.set_ylabel('Throughput Rata-rata (MB/s)')
        ax.set_title(judul_ax)
        ax.set_xticks(x)
        ax.set_xticklabels(label_x)
        ax.legend()

    plt.tight_layout()
    simpan_grafik(nama_file)
    plt.show()


print('Fungsi grafik siap.')

### 6.3 Grafik 4.2.1 — Pesan Teks

In [ ]:
label_teks = ['Pendek (short)', 'Sedang (medium)', 'Panjang (long)']

grafik_waktu_enkripsi_dekripsi(
    df_teks, label_teks,
    'Pesan Teks (UTF-8)',
    'teks_waktu_enkripsi_dekripsi.png'
)

grafik_throughput(
    df_teks, label_teks,
    'Pesan Teks (UTF-8)',
    'teks_throughput.png'
)

### 6.4 Grafik 4.2.2 — File Gambar

In [ ]:
label_gambar = ['Kecil (≤100 KB)', 'Sedang (100 KB-1 MB)', 'Besar (1-5 MB)']

grafik_waktu_enkripsi_dekripsi(
    df_gambar, label_gambar,
    'File Gambar (JPEG/PNG)',
    'gambar_waktu_enkripsi_dekripsi.png'
)

grafik_throughput(
    df_gambar, label_gambar,
    'File Gambar (JPEG/PNG)',
    'gambar_throughput.png'
)

### 6.5 Grafik 4.2.3 — File Audio

In [ ]:
label_audio = ['Kecil (≤1 MB)', 'Sedang (1-5 MB)', 'Besar (5-10 MB)']

grafik_waktu_enkripsi_dekripsi(
    df_audio, label_audio,
    'File Audio (MP3/WAV)',
    'audio_waktu_enkripsi_dekripsi.png'
)

grafik_throughput(
    df_audio, label_audio,
    'File Audio (MP3/WAV)',
    'audio_throughput.png'
)

### 6.6 Grafik 4.2.4 — File Video

In [ ]:
label_video = ['Kecil (≤5 MB)', 'Sedang (5-20 MB)', 'Besar (20-50 MB)']

grafik_waktu_enkripsi_dekripsi(
    df_video, label_video,
    'File Video (MP4)',
    'video_waktu_enkripsi_dekripsi.png'
)

grafik_throughput(
    df_video, label_video,
    'File Video (MP4)',
    'video_throughput.png'
)

## 7. Grafik 4.2.5 — Ringkasan Keseluruhan

In [ ]:
def grafik_tren_keseluruhan(df, nama_file):
    """Grafik garis tren waktu enkripsi vs ukuran data lintas kategori."""
    stat = hitung_statistik(df)
    stat = stat.sort_values('data_size_bytes')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Tren Performa Keseluruhan: AES-GCM-SIV vs AES-GCM',
                 fontsize=13, fontweight='bold', y=1.02)

    for ax, metrik, judul_ax, satuan in [
        (axes[0], 'enc_mean', 'Waktu Enkripsi vs Ukuran Data', 'µs'),
        (axes[1], 'enc_tp_mean', 'Throughput Enkripsi vs Ukuran Data', 'MB/s'),
    ]:
        for algo in ['aes-gcm-siv', 'aes-gcm']:
            sub = stat[stat['algorithm'] == algo].sort_values('data_size_bytes')
            ax.plot(
                sub['data_size_bytes'], sub[metrik],
                marker='o', linewidth=2, markersize=6,
                label=LABELS[algo], color=COLORS[algo]
            )

        ax.set_xlabel('Ukuran Data (bytes)')
        ax.set_ylabel(f'{judul_ax.split(" vs")[0]} ({satuan})')
        ax.set_title(judul_ax)
        ax.legend()
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
        )

    plt.tight_layout()
    simpan_grafik(nama_file)
    plt.show()


grafik_tren_keseluruhan(df, 'ringkasan_tren_keseluruhan.png')

## 8. Tabel Overhead Ukuran Data

In [ ]:
overhead = df.groupby(['algorithm', 'data_type']).agg(
    data_size_bytes=('data_size_bytes', 'first'),
    overhead_bytes=('overhead_bytes', 'first'),
    overhead_pct=('overhead_pct', 'first'),
).reset_index()

print('=== OVERHEAD UKURAN DATA ===')
overhead

## 9. Ekspor Tabel Statistik ke CSV

In [ ]:
stats_df.to_csv('grafik/tabel_statistik_ringkasan.csv', index=False)
tt_enc.to_csv('grafik/tabel_ttest_enkripsi.csv', index=False)
tt_dec.to_csv('grafik/tabel_ttest_dekripsi.csv', index=False)
overhead.to_csv('grafik/tabel_overhead.csv', index=False)

print('Semua tabel berhasil diekspor ke folder grafik/')
print('\nFile yang dihasilkan:')
for f in os.listdir('grafik'):
    print(f'  grafik/{f}')